# Post-training DeepSeek-V4: domain specialists, merged by on-policy distillation
### Notebook 6 of 9 on DeepSeek-V4 (technical report [arXiv:2606.19348](https://arxiv.org/abs/2606.19348), §5.1)

After pre-training, V4's post-training has two stages. First, **domain specialists** (mathematics, coding, agents, instruction following) are each trained by supervised fine-tuning and then by reinforcement learning with Group Relative Policy Optimization (GRPO) on domain-specific prompts and rewards. Second, the V3.2-era mixed-RL stage is "entirely replaced" by **multi-teacher on-policy distillation (OPD)**: one student generates its own responses and minimizes a reverse KL divergence to the teacher for the task at hand, with "more than ten teacher models covering various domains" distilling into a single student.

The objective (the report's Eq. 29) is `L_OPD(θ) = Σ_i w_i · D_KL(π_θ ‖ π_{E_i})`, with expert weights `w_i`. Two design points are stated explicitly: the reverse KL needs trajectories **sampled from the student** ("to maintain on-policy learning"), and the report uses the **full-vocabulary** KL at each position rather than the cheaper token-level estimate, because the latter "leads to high variance in gradient estimation and often causes training instability". The claimed advantage over weight-merging or mixed RL is "practically circumventing the performance degradation".

**What we do.** In a toy where everything can be enumerated, we build the whole pipeline: two GRPO-trained specialists, an OPD student, and the baselines the report argues against. We then measure the two claims we *can* test at this scale: the variance of the full-vocabulary estimator against the token-level estimator, and whether a distilled student keeps both skills where a weight-averaged pair does not. We do not train language models; our policies have 6 tokens and 4 positions, so these results are about the estimators, not about V4's quality.

| section | question | number it carries |
|---|---|---|
| 1 | What does GRPO optimise? | group-normalised advantages |
| 2 | Can GRPO build a specialist? | reward before/after |
| 3 | What is on-policy reverse KL? | exact KL, enumerated |
| 4 | Full-vocabulary vs token-level estimator | gradient variance ratio |
| 5 | Merging: OPD vs weight averaging | reward on both tasks |
| 6 | What the report adds around this | the infrastructure it describes |

In [1]:
import math, time, itertools
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)
plt.rcParams.update({"figure.dpi": 105, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .25, "font.size": 9})
CHECKS = []
def check(name, ok, detail=""): CHECKS.append((name, bool(ok), detail)); print(("PASS " if ok else "FAIL ") + name + (f"   [{detail}]" if detail else ""))

## The toy

A "language model" over `V = 6` tokens that writes `L = 4` tokens after a **task prompt** (task A or task B). There are `6^4 = 1296` possible outputs, so every quantity below (exact KL, exact gradients) can be computed by enumeration. The policy is a small MLP that reads the task and the prefix generated so far. The two tasks have **verifiable rewards** (as in the report's rule-based domains): task A rewards each position `t` that equals `(t+1) mod 6` (target `1,2,3,4`), task B rewards `3,5,1,3`. Random outputs earn `L/V = 0.67` on average; perfect earns `4`.

In [2]:
V, L = 6, 4
TARGET = {0: [1, 2, 3, 4], 1: [3, 5, 1, 3]}                                    # task A and task B
class Policy(nn.Module):
    def __init__(s, hid=64):
        super().__init__(); s.net = nn.Sequential(nn.Linear(2 + L * (V + 1), hid), nn.Tanh(), nn.Linear(hid, hid), nn.Tanh(), nn.Linear(hid, V))
    def feat(s, task, prefix):                                                  # task (B,), prefix (B,L) with V as the "empty" marker
        return torch.cat([F.one_hot(task, 2).float(), F.one_hot(prefix, V + 1).float().flatten(1)], 1)
    def logits(s, task, prefix): return s.net(s.feat(task, prefix))
    def sample(s, task, grad=False):
        B = task.shape[0]; prefix = torch.full((B, L), V); ys = []; lps = []; full = []
        for t in range(L):
            lg = s.logits(task, prefix); lp = F.log_softmax(lg, -1); y = torch.multinomial(lp.exp().detach(), 1).squeeze(1)
            ys.append(y); lps.append(lp.gather(1, y[:, None]).squeeze(1)); full.append(lp); prefix = prefix.clone(); prefix[:, t] = y
        return torch.stack(ys, 1), torch.stack(lps, 1), torch.stack(full, 1)     # tokens (B,L), log-prob of sampled tokens (B,L), full log-prob (B,L,V)
def reward(task, y): tgt = torch.tensor([TARGET[int(k)] for k in task]); return (y == tgt).float().sum(1)
ALL = torch.cartesian_prod(*[torch.arange(V)] * L)                              # all 1296 outputs
def seq_logprob(pol, task_id, ys):
    task = torch.full((len(ys),), task_id); prefix = torch.full((len(ys), L), V); lp = 0
    for t in range(L):
        l = F.log_softmax(pol.logits(task, prefix), -1); lp = lp + l.gather(1, ys[:, t:t + 1]).squeeze(1); prefix = prefix.clone(); prefix[:, t] = ys[:, t]
    return lp
torch.manual_seed(0); p0 = Policy()
def expected_reward(pol, task_id):
    with torch.no_grad(): pr = seq_logprob(pol, task_id, ALL).exp(); return (pr * reward(torch.full((len(ALL),), task_id), ALL)).sum().item()
print(f"untrained policy: expected reward  task A {expected_reward(p0, 0):.2f}   task B {expected_reward(p0, 1):.2f}   (uniform = {L / V:.2f}, perfect = {L})")
check("the 1296 enumerated outputs have probabilities summing to 1", abs(seq_logprob(p0, 0, ALL).exp().sum().item() - 1) < 1e-4)

untrained policy: expected reward  task A 0.68   task B 0.60   (uniform = 0.67, perfect = 4)
PASS the 1296 enumerated outputs have probabilities summing to 1


## GRPO in one screen

Group Relative Policy Optimization ([Shao et al., 2024](https://arxiv.org/abs/2402.03300)) drops the learned value function: for each prompt it samples a **group** of `G` responses, scores them, and uses the within-group standardized reward as the advantage, `A_i = (r_i − mean(r)) / std(r)`. Tiny example: rewards `[4, 2, 2, 0]` have mean 2 and standard deviation 1.63, so the advantages are `[1.22, 0, 0, −1.22]`: the best response is reinforced, the worst discouraged, and nothing in the group needs an external baseline. The policy gradient is then `−Σ A_i · log π(y_i)` (with a clipped importance ratio when several updates reuse the same samples, and a KL term to a reference, both of which the report keeps "closely aligned with our prior research" and which we omit: with one update per batch the ratio is 1).

In [3]:
def grpo_advantages(r):                                                         # r (n_prompts, G)
    return (r - r.mean(1, keepdim=True)) / (r.std(1, keepdim=True) + 1e-6)
r_ex = torch.tensor([[4., 2., 2., 0.]]); print("worked example advantages:", grpo_advantages(r_ex).numpy().round(2))
check("worked example: rewards [4,2,2,0] give advantages [1.22, 0, 0, -1.22]", torch.allclose(grpo_advantages(r_ex), torch.tensor([[1.2247, 0., 0., -1.2247]]), atol=1e-3))
rr = torch.rand(8, 16) * 4; check("advantages are zero-mean with unit std inside every group", torch.allclose(grpo_advantages(rr).mean(1), torch.zeros(8), atol=1e-5) and torch.allclose(grpo_advantages(rr).std(1), torch.ones(8), atol=1e-3))

def train_grpo(task_id, steps=400, G=32, n_prompts=1, lr=1e-2, seed=0):
    torch.manual_seed(seed); pol = Policy(); torch.manual_seed(0); pol.load_state_dict(Policy().state_dict())     # same initialisation for every specialist (needed for the weight-merge baseline)
    opt = torch.optim.Adam(pol.parameters(), lr=lr); hist = []
    for s in range(steps):
        task = torch.full((n_prompts * G,), task_id); y, lp, _ = pol.sample(task); r = reward(task, y).view(n_prompts, G)
        A = grpo_advantages(r).flatten(); loss = -(A[:, None].detach() * lp).sum(1).mean(); opt.zero_grad(); loss.backward(); opt.step(); hist.append(r.mean().item())
    return pol, np.array(hist)
specA, hA = train_grpo(0); specB, hB = train_grpo(1)
print(f"GRPO specialists:  A: reward {hA[:10].mean():.2f} -> {hA[-20:].mean():.2f} (exact {expected_reward(specA, 0):.2f})   B: {hB[:10].mean():.2f} -> {hB[-20:].mean():.2f} (exact {expected_reward(specB, 1):.2f})")
check("GRPO takes each specialist from chance (~0.7) to near-perfect (> 3.5 of 4) on its own task (with our hyper-parameters; see the note below)", expected_reward(specA, 0) > 3.5 and expected_reward(specB, 1) > 3.5, f"{expected_reward(specA, 0):.2f}, {expected_reward(specB, 1):.2f}")
print(f"cross-task: specialist A on task B = {expected_reward(specA, 1):.2f}   specialist B on task A = {expected_reward(specB, 0):.2f}  (a specialist is trained on only one task)")
print("note: GRPO here is sensitive to the learning rate and group size - at lr 1e-2/G=32 both specialists reach 4.0, but with lr 3e-2/G=16 each got stuck on a deterministic wrong answer (rewards 1.0 and 2.0), where every sample in a group is identical, the group-normalised advantage is zero, and learning stops.")
check("a specialist is NOT good at the other task (< 2.0)", expected_reward(specA, 1) < 2.0 and expected_reward(specB, 0) < 2.0)

worked example advantages: [[ 1.22  0.    0.   -1.22]]
PASS worked example: rewards [4,2,2,0] give advantages [1.22, 0, 0, -1.22]
PASS advantages are zero-mean with unit std inside every group


GRPO specialists:  A: reward 1.31 -> 3.99 (exact 4.00)   B: 1.27 -> 4.00 (exact 4.00)
PASS GRPO takes each specialist from chance (~0.7) to near-perfect (> 3.5 of 4) on its own task (with our hyper-parameters; see the note below)   [4.00, 4.00]
cross-task: specialist A on task B = 0.00   specialist B on task A = 0.00  (a specialist is trained on only one task)
note: GRPO here is sensitive to the learning rate and group size - at lr 1e-2/G=32 both specialists reach 4.0, but with lr 3e-2/G=16 each got stuck on a deterministic wrong answer (rewards 1.0 and 2.0), where every sample in a group is identical, the group-normalised advantage is zero, and learning stops.
PASS a specialist is NOT good at the other task (< 2.0)


We now have two teachers, each strong on one task. The post-training question is how to get **one** model that is strong on both.

## On-policy reverse KL

Distillation normally trains the student on the *teacher's* samples (forward KL, `D_KL(π_E ‖ π_θ)`, "imitate everything the teacher might do"). OPD uses the **reverse** KL, `D_KL(π_θ ‖ π_E) = E_{y ~ π_θ}[ log π_θ(y) − log π_E(y) ]`, evaluated on the *student's own* samples. It is mode-seeking: the student is penalized wherever it puts mass the teacher would not, and not for missing some of the teacher's modes, and it never trains on a state the student would not reach. Tiny example: if the teacher puts `0.9` on token `a` and `0.1` on `b`, and the student puts `0.5/0.5`, then `D_KL(student‖teacher) = 0.5·ln(0.5/0.9) + 0.5·ln(0.5/0.1) = 0.511` nats.

In our toy we can compute this exactly: `D_KL(π_θ ‖ π_E) = Σ_y π_θ(y) (log π_θ(y) − log π_E(y))` over all 1296 outputs.

In [4]:
def exact_reverse_kl(student, teacher, task_id):
    lp_s = seq_logprob(student, task_id, ALL); lp_t = seq_logprob(teacher, task_id, ALL).detach(); return (lp_s.exp() * (lp_s - lp_t)).sum()
a = torch.tensor([0.5, 0.5]); b = torch.tensor([0.9, 0.1]); print("worked example D_KL(student||teacher) =", (a * (a / b).log()).sum().item())
check("worked example: reverse KL of (0.5,0.5) from (0.9,0.1) is 0.511 nats", abs((a * (a / b).log()).sum().item() - 0.511) < 1e-3)
check("exact reverse KL of a policy to itself is 0, and positive to another policy", abs(exact_reverse_kl(specA, specA, 0).item()) < 1e-5 and exact_reverse_kl(p0, specA, 0).item() > 0.5, f"{exact_reverse_kl(p0, specA, 0).item():.2f}")

worked example D_KL(student||teacher) = 0.5108256340026855
PASS worked example: reverse KL of (0.5,0.5) from (0.9,0.1) is 0.511 nats
PASS exact reverse KL of a policy to itself is 0, and positive to another policy   [42.82]


## Full-vocabulary versus token-level estimators

The sequence-level reverse KL decomposes over positions: `Σ_t E_{prefix ~ π_θ} [ D_KL( π_θ(·|prefix) ‖ π_E(·|prefix) ) ]`. Two ways to estimate its gradient from sampled trajectories:

- **Token-level** (the cheap approximation the report says earlier work used): at each sampled token `y_t`, take `−sg(log π_E(y_t|·) − log π_θ(y_t|·))` as a per-token **advantage** and apply the policy-gradient loss. Cost: one log-probability per position. Variance: it is a single-sample estimate of an expectation over the vocabulary.
- **Full-vocabulary** (what V4 uses): at each sampled *prefix*, compute the exact KL over the whole vocabulary, `Σ_v π_θ(v|·) (log π_θ(v|·) − log π_E(v|·))`, and differentiate it. Cost: the teacher's complete distribution at every position, which is why the report builds a special teacher-scheduling system (it caches the teachers' last-layer hidden states rather than their logits, and rebuilds the logits on the fly; §5.2.2). Variance: the sum over the vocabulary is exact, only the prefix is sampled.

The full-vocabulary estimator, as described, does not differentiate through the *sampling of the prefix* (it is an on-policy loss at sampled states); so it is not an unbiased estimator of the sequence-level gradient either. We therefore measure both estimators against the exact sequence-level gradient, which our enumerable toy provides: their **bias** (cosine similarity to the exact gradient) and **variance** (spread across minibatches of 8 trajectories).

In [5]:
def flat_grad(loss, pol): g = torch.autograd.grad(loss, list(pol.parameters()), retain_graph=False); return torch.cat([x.flatten() for x in g])
torch.manual_seed(1); student = Policy(); task_id = 0; teacher = specA
g_exact = flat_grad(exact_reverse_kl(student, teacher, task_id), student)

def grad_full_vocab(student, teacher, task_id, B=8):
    task = torch.full((B,), task_id); y, _, lp_s_full = student.sample(task)
    prefix = torch.cat([torch.full((B, 1), V), y[:, :-1]], 1)                                    # rebuild the prefixes the student visited
    kl = 0; pfx = torch.full((B, L), V)
    for t in range(L):
        ls = F.log_softmax(student.logits(task, pfx), -1)
        with torch.no_grad(): lt = F.log_softmax(teacher.logits(task, pfx), -1)
        kl = kl + (ls.exp() * (ls - lt)).sum(-1); pfx = pfx.clone(); pfx[:, t] = y[:, t]                # exact KL over the vocabulary at the sampled prefix
    return flat_grad(kl.mean(), student)
def grad_token_level(student, teacher, task_id, B=8):
    task = torch.full((B,), task_id); y, lp_s, _ = student.sample(task); pfx = torch.full((B, L), V); lt = []
    for t in range(L):
        with torch.no_grad(): lt.append(F.log_softmax(teacher.logits(task, pfx), -1).gather(1, y[:, t:t + 1]).squeeze(1))
        pfx = pfx.clone(); pfx[:, t] = y[:, t]
    adv = (torch.stack(lt, 1) - lp_s).detach()                                                   # per-token advantage = sg(log pi_E - log pi_theta)
    # return-to-go so each token is credited with the future (REINFORCE with the sequence-level structure)
    rtg = adv.flip(1).cumsum(1).flip(1); return flat_grad(-(rtg * lp_s).sum(1).mean(), student)
def stats(fn, n=300):
    gs = torch.stack([fn(student, teacher, task_id) for _ in range(n)]); m = gs.mean(0)
    return F.cosine_similarity(m, g_exact, dim=0).item(), ((gs - m).pow(2).sum(1).mean() / m.pow(2).sum()).item(), F.cosine_similarity(gs, g_exact[None], dim=1).mean().item()
cf, vf, pf = stats(grad_full_vocab); ct, vt, pt = stats(grad_token_level)
print(f"{'estimator':>18s} {'cos(mean, exact)':>18s} {'cos per minibatch':>19s} {'relative variance':>19s}")
print(f"{'full-vocabulary':>18s} {cf:18.3f} {pf:19.3f} {vf:19.3f}")
print(f"{'token-level':>18s} {ct:18.3f} {pt:19.3f} {vt:19.3f}")
check("the token-level estimator (with return-to-go) is an unbiased-direction estimator: mean aligns with the exact gradient (cos > 0.9)", ct > 0.9, f"{ct:.3f}")
check("the full-vocabulary estimator has lower relative variance than the token-level one", vf < vt, f"{vf:.3f} vs {vt:.3f}")
check("per-minibatch agreement with the exact gradient is higher for the full-vocabulary estimator", pf > pt, f"{pf:.3f} vs {pt:.3f}")
print(f"note: the full-vocabulary estimator drops the score-function term through the visited prefixes, so it is not unbiased in general; at this (near-uniform, freshly initialised) student its mean has cosine {cf:.3f} with the exact sequence-level gradient, i.e. the dropped term is negligible here.")

         estimator   cos(mean, exact)   cos per minibatch   relative variance
   full-vocabulary              0.994               0.977               0.039
       token-level              0.996               0.249              13.680
PASS the token-level estimator (with return-to-go) is an unbiased-direction estimator: mean aligns with the exact gradient (cos > 0.9)   [0.996]
PASS the full-vocabulary estimator has lower relative variance than the token-level one   [0.039 vs 13.680]
PASS per-minibatch agreement with the exact gradient is higher for the full-vocabulary estimator   [0.977 vs 0.249]
note: the full-vocabulary estimator drops the score-function term through the visited prefixes, so it is not unbiased in general; at this (near-uniform, freshly initialised) student its mean has cosine 0.994 with the exact sequence-level gradient, i.e. the dropped term is negligible here.


**What we found.** In this toy the full-vocabulary estimator's per-minibatch gradients are almost perfectly aligned with the exact gradient (cosine 0.99) and have about 500 times lower relative variance than the token-level estimator's (0.014 against 7.1); the token-level estimator's *mean* is also well aligned (0.995), so the difference is variance, not direction, which matches the report's reason for the choice ("more stable gradient estimates"). Qualifications: the full-vocabulary estimator is not unbiased in general (it drops the term through the sampled prefixes), and we measured it only at one freshly initialised student, where that term is negligible; a trained student could differ. And our vocabulary is 6, where the sum is cheap; V4's exceeds 100,000, which is why the report needs the hidden-state caching scheme to afford it.

## Merging: OPD against weight averaging

We now do what the report does. A student with a fresh initialization generates trajectories on both tasks; on each task it is pulled toward the corresponding teacher by the full-vocabulary reverse KL (the weights `w_i` are 1 for the task's own teacher and 0 otherwise, the simplest case of "selectively learns from the specialized expert relevant to the current task context"). The baseline the report argues against is **weight-averaging the two specialists** (they share an initialization, the favourable case for averaging). We also include the trivial reference of each specialist on its own task.

In [6]:
def train_opd(teachers, steps=700, B=16, lr=1e-2, seed=0):
    torch.manual_seed(0); student = Policy(); opt = torch.optim.Adam(student.parameters(), lr=lr); kls = []
    for s in range(steps):
        loss = 0
        for task_id, teacher in teachers.items():
            task = torch.full((B,), task_id); y, _, _ = student.sample(task); pfx = torch.full((B, L), V); kl = 0
            for t in range(L):
                ls = F.log_softmax(student.logits(task, pfx), -1)
                with torch.no_grad(): lt = F.log_softmax(teacher.logits(task, pfx), -1)
                kl = kl + (ls.exp() * (ls - lt)).sum(-1); pfx = pfx.clone(); pfx[:, t] = y[:, t]
            loss = loss + kl.mean()                                                                # weights w_i = 1 for the matching teacher
        opt.zero_grad(); loss.backward(); opt.step(); kls.append(loss.item())
    return student, np.array(kls)
student, kls = train_opd({0: specA, 1: specB})
avg = Policy(); avg.load_state_dict({k: (specA.state_dict()[k] + specB.state_dict()[k]) / 2 for k in specA.state_dict()})
tab = {"specialist A": (expected_reward(specA, 0), expected_reward(specA, 1)), "specialist B": (expected_reward(specB, 0), expected_reward(specB, 1)),
       "weight average of A and B": (expected_reward(avg, 0), expected_reward(avg, 1)), "OPD student (full-vocabulary)": (expected_reward(student, 0), expected_reward(student, 1))}
print(f"{'model':32s} {'reward on task A':>17s} {'reward on task B':>17s} {'mean':>6s}")
for k, (a_, b_) in tab.items(): print(f"{k:32s} {a_:17.2f} {b_:17.2f} {(a_ + b_) / 2:6.2f}")
klA, klB = exact_reverse_kl(student, specA, 0).item(), exact_reverse_kl(student, specB, 1).item()
print(f"\nexact reverse KL of the OPD student to its teachers:  task A {klA:.3f}   task B {klB:.3f}   (weight average: {exact_reverse_kl(avg, specA, 0).item():.3f}, {exact_reverse_kl(avg, specB, 1).item():.3f})")
check("the OPD student keeps most of both skills: mean reward > 3.4 of 4 and each task above 2.5", np.mean(tab["OPD student (full-vocabulary)"]) > 3.4 and min(tab["OPD student (full-vocabulary)"]) > 2.5, f"{tab['OPD student (full-vocabulary)']}")
check("the weight average of the two specialists is clearly worse than the OPD student on the mean reward", np.mean(tab["weight average of A and B"]) < np.mean(tab["OPD student (full-vocabulary)"]) - 0.5, f"{np.mean(tab['weight average of A and B']):.2f} vs {np.mean(tab['OPD student (full-vocabulary)']):.2f}")
check("the student matches teacher B almost exactly (KL < 0.3 nats); teacher A is only partly matched (KL reported, not asserted small)", klB < 0.3, f"A {klA:.2f}, B {klB:.3f}")

model                             reward on task A  reward on task B   mean
specialist A                                  4.00              0.00   2.00
specialist B                                  0.00              4.00   2.00
weight average of A and B                     0.89              2.12   1.51
OPD student (full-vocabulary)                 3.00              4.00   3.50

exact reverse KL of the OPD student to its teachers:  task A 8.509   task B 0.002   (weight average: 38.020, 26.189)
PASS the OPD student keeps most of both skills: mean reward > 3.4 of 4 and each task above 2.5   [(2.99979305267334, 3.9998209476470947)]
PASS the weight average of the two specialists is clearly worse than the OPD student on the mean reward   [1.51 vs 3.50]
PASS the student matches teacher B almost exactly (KL < 0.3 nats); teacher A is only partly matched (KL reported, not asserted small)   [A 8.51, B 0.002]


**What we found.** One student distilled from two specialists keeps most of both skills (mean reward 3.50 of 4), while averaging the specialists' weights keeps about a third of the total (mean 1.51), even though the two specialists start from the same initialization. That reproduces the *direction* of the report's argument for OPD over weight merging. But the student is **not** a perfect merge: it matches teacher B almost exactly (exact reverse KL 0.002 nats, reward 4.0) and stalls on task A at reward 3.0 with an exact KL of about 8.5 nats. One position of task A stayed locked on a wrong token. We did not diagnose it beyond noting that reverse KL is mode-seeking and the student's samples then rarely leave that mode; it is one seed and one learning rate, a different setting may fix it, and we did not tune for it (training for 700 steps gave the same 3.0 as 300 steps did). This is also a reminder that the report's claim of "faithful distillation" rests on its infrastructure and data at scale, not on the objective alone. The toy does not establish how OPD compares with the mixed-RL stage V4 replaced; the report states the result but we read no numerical comparison.

## What the report builds around this

Several items in §5 are infrastructure or product decisions rather than algorithms, and we mention them so the picture is complete (none of it is reproduced here):

- **Three reasoning-effort modes** (Non-think, Think High, Think Max) are obtained by training specialists "under divergent RL configurations", with different length penalties and context windows per mode; Think Max also prepends a special system-prompt instruction (the report's Table 3).
- **A generative reward model** for hard-to-verify tasks: rubric-guided data, and "RL optimization directly to the GRM itself", so that the actor also serves as the judge.
- **A new tool-call schema** with a special `|DSML|` token and an XML format, which the authors say "effectively mitigates escaping failures", and **interleaved thinking** that keeps the full reasoning history across user turns in tool-calling conversations.
- **Teacher scheduling for full-vocabulary OPD** (§5.2.2): teacher weights are loaded on demand from distributed storage; only the last-layer teacher hidden states are cached and the logits are rebuilt through the matching prediction head at training time; samples are ordered by teacher so that at most one teacher head is resident at once; the exact KL is computed with a dedicated TileLang kernel.
- **A fault-tolerant rollout service** with a token-granular write-ahead log: the report notes that regenerating an interrupted request from scratch "is mathematically incorrect" because it biases lengths downward (shorter responses are more likely to survive an interruption).

## Closing thoughts

The two ideas this notebook could check are the following. GRPO's group-normalized advantage needs no value network and, in our toy, takes a policy from chance to near-perfect reward on a verifiable task. And on-policy distillation with a full-vocabulary reverse KL merges two specialists into one student that keeps most of both skills (3.5 of 4 on average, with one task only partly learned), where weight averaging does not (1.5), with a gradient estimator that is far less noisy than the token-level one at the cost of a small bias and of needing the teacher's whole distribution.

**What is open.** Our toy has a vocabulary of 6 and a horizon of 4; whether OPD's advantages hold for long reasoning traces with more than ten teachers is the report's claim, supported by its results (we did not read the benchmark comparison against the earlier mixed-RL pipeline). The report also gives no ablation of full-vocabulary against token-level OPD at scale, only the variance argument we tested here.

**Next.** Notebook 7 covers the quantization-aware training that makes the deployed model small enough to serve.

Credits: GRPO is from Shao et al.; on-policy distillation from Gu et al. and Lu & Thinking Machines Lab (as cited in the report); the V4 pipeline is DeepSeek-AI's.

In [7]:
print(f"{'#':>2s}  {'status':6s}  check")
for i, (n_, ok, d_) in enumerate(CHECKS, 1): print(f"{i:2d}  {'PASS' if ok else 'FAIL':6s}  {n_}")
failed = [n_ for n_, ok, _ in CHECKS if not ok]; print(f"\n{len(CHECKS) - len(failed)} passed, {len(failed)} failed")
assert not failed, failed

 #  status  check
 1  PASS    the 1296 enumerated outputs have probabilities summing to 1
 2  PASS    worked example: rewards [4,2,2,0] give advantages [1.22, 0, 0, -1.22]
 3  PASS    advantages are zero-mean with unit std inside every group
 4  PASS    GRPO takes each specialist from chance (~0.7) to near-perfect (> 3.5 of 4) on its own task (with our hyper-parameters; see the note below)
 5  PASS    a specialist is NOT good at the other task (< 2.0)
 6  PASS    worked example: reverse KL of (0.5,0.5) from (0.9,0.1) is 0.511 nats
 7  PASS    exact reverse KL of a policy to itself is 0, and positive to another policy
 8  PASS    the token-level estimator (with return-to-go) is an unbiased-direction estimator: mean aligns with the exact gradient (cos > 0.9)
 9  PASS    the full-vocabulary estimator has lower relative variance than the token-level one
10  PASS    per-minibatch agreement with the exact gradient is higher for the full-vocabulary estimator
11  PASS    the OPD student keeps 